In [12]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ✅ File path
file_path = "/content/drive/MyDrive/ML_OPS/nobroker_cleaned_data.csv"

# ✅ Load data
df = pd.read_csv(file_path)

# ✅ Drop columns with all missing values
df = df.dropna(axis=1, how='all')

# ✅ Separate features and target
X = df.iloc[:, :-1]  # All columns except the last one
y = df.iloc[:, -1]   # Last column as target

# ✅ Separate numeric and categorical columns
numeric_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

# ✅ Preprocessing steps
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# ✅ Combine preprocessors
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# ✅ Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ✅ Define models
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Support Vector Regressor": SVR(),
    "XGBoost Regressor": XGBRegressor(objective='reg:squarederror', random_state=42)
}

# ✅ Evaluate models
results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    start = time.time()
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    end = time.time()

    mape = np.mean(np.abs((y_test - y_pred) / np.clip(y_test, a_min=1e-8, a_max=None))) * 100

    results.append({
        "Model": name,
        "R² Score": round(r2_score(y_test, y_pred), 4),
        "MAE": round(mean_absolute_error(y_test, y_pred), 4),
        "MAPE (%)": round(mape, 2),
        "Train Time (s)": round(end - start, 3)
    })

# ✅ Show results
results_df = pd.DataFrame(results).sort_values(by="MAE")
print("\n📊 Model Evaluation Summary:")
print(results_df.to_string(index=False))


📊 Model Evaluation Summary:
                   Model  R² Score    MAE  MAPE (%)  Train Time (s)
       XGBoost Regressor    0.9943 0.4091      2.43           0.102
           Decision Tree    0.9898 0.5958      3.67           0.018
       Linear Regression    0.9855 0.7346      4.27           0.038
           Random Forest    0.9623 1.2424      4.92           0.163
        Ridge Regression    0.9742 1.2624      6.69           0.020
        Lasso Regression    0.8859 2.6009     12.62           0.020
Support Vector Regressor   -0.1017 5.3734     17.36           0.022


In [10]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
